In [63]:
import pandas as pd
import datetime

In [64]:
data = pd.read_csv('LECLERC_cleaned.csv')

In [65]:
data = data.loc[data['Product Name'] != 'Non trouvé']

In [66]:
def calculate_days(row):
    date_range = row['Delivery Date']
    date_range = date_range.replace('Prévue entre le ', '')
    date_range = date_range.replace('Prévue le ', '')
    if ' et le ' in date_range:
        max_date = date_range.split(' et le ')[-1]
    else:
        max_date = date_range
    max_date_dt = datetime.datetime.strptime(max_date, '%d/%m/%y')
    scrap_date = datetime.datetime.strptime(row['Timestamp'], '%d/%m/%Y %H:%M:%S')
    scrap_date = datetime.datetime.combine(scrap_date.date(), datetime.time(0, 0))
    
    difference = (max_date_dt - scrap_date).days
    
    return difference

In [67]:
import numpy as np

def calculate_experience(row):
    if not pd.isna(row['SellerActivityDate']):
        start = str(row['SellerActivityDate'])
        start_dt = datetime.datetime.strptime(start, '%d/%m/%Y')
        try:
            scrap_date = datetime.datetime.strptime(row['Timestamp'], '%d/%m/%Y %H:%M:%S')
        except ValueError as e:
            print(f"Error parsing timestamp: {row['Timestamp']}")
            raise e
        
        scrap_date = scrap_date.date()
        start_dt = start_dt.date()
        
        difference = (scrap_date - start_dt).days
    else:
        difference = np.nan
    
    return difference 



In [68]:
data['shipping_days'] = data.apply(calculate_days, axis=1)

Ajout des infos vendeurs 

In [69]:
data.drop(columns = ['Seller Rating','Delivery Date','Platform'],inplace=True)

In [70]:
seller = pd.read_csv('Sellers.csv')
seller.drop(columns = ['SellerStatus'],inplace=True)

In [71]:
data_merge = pd.merge(data,seller,how='left',on='Seller')

In [72]:
data_merge.head()

,Product Name,Seller,Price,Delivery Fees,Product State,Timestamp,ID,shipping_days,Seller Rating,NbSellerRatings,SellerActivityDate,SellerCountry
0,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1349.00,Offerte,NEUF,24/12/2024 13:30:11,160f282c-11af-427d-8ea5-a98d1badba9c,4,NaN,NaN,NaN,France
1,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Icoza,1408.17,Offerte,NEUF,24/12/2024 13:30:11,98b12821-acb5-4aa6-a5d4-129db155e1d9,4,4.0,140.0,11/09/2020,France
2,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Stock e-commerce,1409.17,Offerte,NEUF,24/12/2024 13:30:11,5ace8dfe-bef8-428f-9a20-eda099c4661a,4,4.0,242.0,17/06/2021,France
3,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Maxmovil,1432.28,Offerte,NEUF,24/12/2024 13:30:11,5e16f708-0b47-4d88-9e49-ef4d3dd29796,4,4.0,14.0,25/06/2021,Espagne
4,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Monsieurplus,1473.99,Offerte,NEUF,24/12/2024 13:30:11,d282bc92-68ff-4039-9239-32e2443a6a19,4,4.0,7.0,20/12/2022,France


In [73]:
data_merge.isna().sum()/len(data_merge)

Product Name          0.000000
Seller                0.000000
Price                 0.000000
Delivery Fees         0.000000
Product State         0.000000
Timestamp             0.000000
ID                    0.000000
shipping_days         0.000000
Seller Rating         0.101317
NbSellerRatings       0.101317
SellerActivityDate    0.101317
SellerCountry         0.000000
dtype: float64

In [74]:
data_merge[data_merge['SellerActivityDate'].apply(lambda x: isinstance(x, float))].head(3)

,Product Name,Seller,Price,Delivery Fees,Product State,Timestamp,ID,shipping_days,Seller Rating,NbSellerRatings,SellerActivityDate,SellerCountry
0,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1349.0,Offerte,NEUF,24/12/2024 13:30:11,160f282c-11af-427d-8ea5-a98d1badba9c,4,NaN,NaN,NaN,France
9,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1098.0,Offerte,NEUF,24/12/2024 13:30:17,3daccedd-2936-4a16-bb57-abf5e372681c,4,NaN,NaN,NaN,France
27,"Apple iPhone 16 Plus 17 cm (6.7"") Double SIM i...",E.Leclerc,1240.0,Offerte,NEUF,24/12/2024 13:30:38,f84f0fa9-55f1-43cd-a856-05f54b591ea7,4,NaN,NaN,NaN,France


In [75]:
data_merge['SellerExperience'] = data_merge.apply(calculate_experience, axis=1)

In [76]:
data_merge.isna().sum()

Product Name              0
Seller                    0
Price                     0
Delivery Fees             0
Product State             0
Timestamp                 0
ID                        0
shipping_days             0
Seller Rating         20622
NbSellerRatings       20622
SellerActivityDate    20622
SellerCountry             0
SellerExperience      20622
dtype: int64

Suppression de delivery fees car sont tous = offerte

In [77]:
columns_to_delete = ['Delivery Fees','SellerActivityDate','Product State']

In [78]:
#Supprimer la catégorie d'état de téléphone la plus fréquente comme variable de reférence
len(data_merge[data_merge['Product State'] == 'NEUF']['ID'].unique())/len(data_merge['ID'].unique())

0.9172413793103448

In [79]:
dummies_state = pd.get_dummies(data_merge['Product State'], drop_first=True)
data_merge = pd.concat([data_merge, dummies_state], axis=1)

In [80]:
data_merge['SellerCountry'].unique()

array(['France', 'Espagne', 'Luxembourg', 'Lettonie', 'Italie',
       'France       '], dtype=object)

In [81]:
data_merge['SellerCountry'].replace({
    'France       ':'France'
},inplace=True)

C:\Users\zoero\AppData\Local\Temp\ipykernel_17556\2858434610.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_merge['SellerCountry'].replace({


In [82]:
dummies_country = pd.get_dummies(data_merge['SellerCountry'])
dummies_country.drop('France', axis=1, inplace=True)
data_merge = pd.concat([data_merge, dummies_country], axis=1)

In [83]:
data_merge.head()

,Product Name,Seller,Price,Delivery Fees,Product State,Timestamp,ID,shipping_days,Seller Rating,NbSellerRatings,...,SellerExperience,OCCASION - BON ÉTAT,OCCASION - EXCELLENT ÉTAT,OCCASION - PARFAIT - JAMAIS UTILISÉ,OCCASION - TRÉS BON ÉTAT,OCCASION - ÉTAT CORRECT,Espagne,Italie,Lettonie,Luxembourg
0,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1349.00,Offerte,NEUF,24/12/2024 13:30:11,160f282c-11af-427d-8ea5-a98d1badba9c,4,NaN,NaN,...,NaN,False,False,False,False,False,False,False,False,False
1,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Icoza,1408.17,Offerte,NEUF,24/12/2024 13:30:11,98b12821-acb5-4aa6-a5d4-129db155e1d9,4,4.0,140.0,...,1565.0,False,False,False,False,False,False,False,False,False
2,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Stock e-commerce,1409.17,Offerte,NEUF,24/12/2024 13:30:11,5ace8dfe-bef8-428f-9a20-eda099c4661a,4,4.0,242.0,...,1286.0,False,False,False,False,False,False,False,False,False
3,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Maxmovil,1432.28,Offerte,NEUF,24/12/2024 13:30:11,5e16f708-0b47-4d88-9e49-ef4d3dd29796,4,4.0,14.0,...,1278.0,False,False,False,False,False,True,False,False,False
4,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Monsieurplus,1473.99,Offerte,NEUF,24/12/2024 13:30:11,d282bc92-68ff-4039-9239-32e2443a6a19,4,4.0,7.0,...,735.0,False,False,False,False,False,False,False,False,False


In [84]:
data_merge.drop(columns = columns_to_delete,inplace=True)

In [85]:
data_merge.isna().sum()

Product Name                               0
Seller                                     0
Price                                      0
Timestamp                                  0
ID                                         0
shipping_days                              0
Seller Rating                          20622
NbSellerRatings                        20622
SellerCountry                              0
SellerExperience                       20622
OCCASION - BON ÉTAT                        0
OCCASION - EXCELLENT ÉTAT                  0
OCCASION - PARFAIT - JAMAIS UTILISÉ        0
OCCASION - TRÉS BON ÉTAT                   0
OCCASION - ÉTAT CORRECT                    0
Espagne                                    0
Italie                                     0
Lettonie                                   0
Luxembourg                                 0
dtype: int64

Fillna des téléphones vendus par Leclerc par la moyenne des autres 

In [86]:
mean_nb_rating = data_merge['NbSellerRatings'].mean()
mean_nb_rating

np.float64(79.60978476576808)

In [87]:
median_rating = data_merge['Seller Rating'].median()
data_merge['Seller Rating'].fillna(median_rating, inplace=True)
data_merge['NbSellerRatings'].fillna(mean_nb_rating, inplace=True)

C:\Users\zoero\AppData\Local\Temp\ipykernel_17556\2506381378.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_merge['Seller Rating'].fillna(median_rating, inplace=True)
C:\Users\zoero\AppData\Local\Temp\ipykernel_17556\2506381378.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a

In [88]:
median_seller_experience = data_merge['SellerExperience'].median()
data_merge['SellerExperience'].fillna(median_seller_experience,inplace=True)

C:\Users\zoero\AppData\Local\Temp\ipykernel_17556\1889923872.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_merge['SellerExperience'].fillna(median_seller_experience,inplace=True)


In [89]:
data_merge.drop(columns=['SellerCountry'],inplace=True)

In [90]:
data_merge.isna().sum()

Product Name                           0
Seller                                 0
Price                                  0
Timestamp                              0
ID                                     0
shipping_days                          0
Seller Rating                          0
NbSellerRatings                        0
SellerExperience                       0
OCCASION - BON ÉTAT                    0
OCCASION - EXCELLENT ÉTAT              0
OCCASION - PARFAIT - JAMAIS UTILISÉ    0
OCCASION - TRÉS BON ÉTAT               0
OCCASION - ÉTAT CORRECT                0
Espagne                                0
Italie                                 0
Lettonie                               0
Luxembourg                             0
dtype: int64

In [91]:
boolean_variables = [                         
'OCCASION - BON ÉTAT',                    
'OCCASION - EXCELLENT ÉTAT',              
'OCCASION - PARFAIT - JAMAIS UTILISÉ',    
'OCCASION - TRÉS BON ÉTAT',               
'OCCASION - ÉTAT CORRECT' ,               
'Espagne'        ,                        
'Italie'        ,                         
'Lettonie' ,                              
'Luxembourg']

In [92]:
data_merge.dtypes

Product Name                            object
Seller                                  object
Price                                  float64
Timestamp                               object
ID                                      object
shipping_days                            int64
Seller Rating                          float64
NbSellerRatings                        float64
SellerExperience                       float64
OCCASION - BON ÉTAT                       bool
OCCASION - EXCELLENT ÉTAT                 bool
OCCASION - PARFAIT - JAMAIS UTILISÉ       bool
OCCASION - TRÉS BON ÉTAT                  bool
OCCASION - ÉTAT CORRECT                   bool
Espagne                                   bool
Italie                                    bool
Lettonie                                  bool
Luxembourg                                bool
dtype: object

In [93]:
for c in boolean_variables:
    data_merge[c].replace({
        True:1,
        False:0
    },inplace=True)

C:\Users\zoero\AppData\Local\Temp\ipykernel_17556\3291783102.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_merge[c].replace({
C:\Users\zoero\AppData\Local\Temp\ipykernel_17556\3291783102.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_merge[c].replace({


In [94]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# # Créer le heatmap
# plt.figure(figsize=(10, 8))  # Taille de la figure
# sns.heatmap(data_merge.drop(columns=['Product Name','Timestamp','ID','Seller']), annot=True, cmap='coolwarm')  # 'annot=True' pour afficher les valeurs dans les cases
# plt.title('Heatmap Example')  # Ajouter un titre
# plt.show()  # Afficher le graphique


We keep products that were available
for sale in a stable manner over three consecutive days (and not out of stock)

In [95]:
data_merge.head()

,Product Name,Seller,Price,Timestamp,ID,shipping_days,Seller Rating,NbSellerRatings,SellerExperience,OCCASION - BON ÉTAT,OCCASION - EXCELLENT ÉTAT,OCCASION - PARFAIT - JAMAIS UTILISÉ,OCCASION - TRÉS BON ÉTAT,OCCASION - ÉTAT CORRECT,Espagne,Italie,Lettonie,Luxembourg
0,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1349.00,24/12/2024 13:30:11,160f282c-11af-427d-8ea5-a98d1badba9c,4,4.0,79.609785,1311.0,0,0,0,0,0,0,0,0,0
1,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Icoza,1408.17,24/12/2024 13:30:11,98b12821-acb5-4aa6-a5d4-129db155e1d9,4,4.0,140.000000,1565.0,0,0,0,0,0,0,0,0,0
2,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Stock e-commerce,1409.17,24/12/2024 13:30:11,5ace8dfe-bef8-428f-9a20-eda099c4661a,4,4.0,242.000000,1286.0,0,0,0,0,0,0,0,0,0
3,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Maxmovil,1432.28,24/12/2024 13:30:11,5e16f708-0b47-4d88-9e49-ef4d3dd29796,4,4.0,14.000000,1278.0,0,0,0,0,0,1,0,0,0
4,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Monsieurplus,1473.99,24/12/2024 13:30:11,d282bc92-68ff-4039-9239-32e2443a6a19,4,4.0,7.000000,735.0,0,0,0,0,0,0,0,0,0


In [96]:
from datetime import timedelta

# Trier les données
df = data_merge.sort_values(by=['ID', 'Timestamp'])
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M:%S')

# Filtrer les données à partir du 26/12
start_date = pd.to_datetime('26/12/2024')
df = df[df['Timestamp'] >= start_date]

# Trier les données
df = df.sort_values(by=['ID', 'Timestamp'])

def filter_and_identify_sequences(group):
    group = group.sort_values(by='Timestamp')
    group['time_diff'] = group['Timestamp'].diff().fillna(pd.Timedelta(seconds=0))
    breaks = group['time_diff'] > timedelta(hours=24)
    group['block'] = breaks.cumsum()

    # Séparer les blocs valides et non valides
    valid_blocks = group.groupby('block').filter(lambda x: (x['Timestamp'].max() - x['Timestamp'].min()) >= timedelta(days=2))
    non_valid_blocks = group[~group['block'].isin(valid_blocks['block'].unique())]

    return valid_blocks, non_valid_blocks

# Appliquer la fonction à chaque groupe ID
results = df.groupby('ID').apply(lambda g: filter_and_identify_sequences(g))

# Extraire les résultats
valid_data = pd.concat([result[0] for result in results])
non_valid_data = pd.concat([result[1] for result in results])

# Afficher ou utiliser valid_data et non_valid_data
print("Valid Data (conserved):")
print(valid_data)
print("\nNon-Valid Data (deleted):")
print(non_valid_data)

C:\Users\zoero\AppData\Local\Temp\ipykernel_17556\3051832962.py:8: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  start_date = pd.to_datetime('26/12/2024')
C:\Users\zoero\AppData\Local\Temp\ipykernel_17556\3051832962.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results = df.groupby('ID').apply(lambda g: filter_and_identify_sequences(g))


Valid Data (conserved):
                                             Product Name              Seller  \
127     Apple iPhone 16 Plus 17 cm (6.7") Double SIM i...         ESSEBI SHOP   
221     Apple iPhone 16 Plus 17 cm (6.7") Double SIM i...         ESSEBI SHOP   
315     Apple iPhone 16 Plus 17 cm (6.7") Double SIM i...         ESSEBI SHOP   
409     Apple iPhone 16 Plus 17 cm (6.7") Double SIM i...         ESSEBI SHOP   
503     Apple iPhone 16 Plus 17 cm (6.7") Double SIM i...         ESSEBI SHOP   
...                                                   ...                 ...   
203098  Apple iPhone 16 Plus 17 cm (6.7") Double SIM i...  La Boutique du Net   
203196  Apple iPhone 16 Plus 17 cm (6.7") Double SIM i...  La Boutique du Net   
203294  Apple iPhone 16 Plus 17 cm (6.7") Double SIM i...  La Boutique du Net   
203391  Apple iPhone 16 Plus 17 cm (6.7") Double SIM i...  La Boutique du Net   
203489  Apple iPhone 16 Plus 17 cm (6.7") Double SIM i...  La Boutique du Net   

   

In [97]:
len(valid_data)

201222

In [98]:
len(data_merge)

203539

In [99]:
len(non_valid_data)

2223

In [100]:
data_merge['Timestamp'] = pd.to_datetime(data_merge['Timestamp'], format='%d/%m/%Y %H:%M:%S')
data_merge['key'] = data_merge['ID'].astype(str) + data_merge['Timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
non_valid_data['key'] = non_valid_data['ID'].astype(str) + non_valid_data['Timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Créer un set des clés de non_valid_data pour une recherche rapide
keys_to_remove = set(non_valid_data['key'])
len(keys_to_remove)
data_merge['to_remove'] = data_merge['key'].isin(keys_to_remove)

# Filtrer df pour conserver uniquement les enregistrements non marqués
cleaned_df = data_merge[~data_merge['to_remove']].drop(columns=['key', 'to_remove'])  # Supprimer les colonnes temporaires


In [101]:
cleaned_df.to_csv('LECLERC_ordered.csv')